## 1. Imports and Setup
We begin by importing PyTorch for deep learning and scikit-learn for the specific normalization techniques required by the research paper.

In [11]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

# PyTorch Imports
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

# Feature Engineering
from sklearn.preprocessing import StandardScaler

# Configuration
DATA_PROCESSED = Path("../data/processed")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

Using device: cpu


## 2. Load Preprocessed Data
We load the `train_clean`, `test_clean`, and `feature_map` artifacts created in the preprocessing notebook. This ensures we are working with the validated, sequence-filtered dataset.

In [12]:
# Load the cleaned datasets
with open(DATA_PROCESSED / "train_clean.pkl", "rb") as f:
    train_df = pickle.load(f)

with open(DATA_PROCESSED / "test_clean.pkl", "rb") as f:
    test_df = pickle.load(f)

with open(DATA_PROCESSED / "feature_map.pkl", "rb") as f:
    feature_map = pickle.load(f)

# Extract feature column names for easy access
IMU_COLS = feature_map["imu_cols"]
THM_COLS = feature_map["thm_cols"]
TOF_COLS = feature_map["tof_cols"]
LABEL_COL = feature_map["label_col"]

# Create a mapping from label strings to integers
# The paper uses 18 classes (8 BFRBs, 10 Non-BFRBs)
unique_labels = sorted(train_df[LABEL_COL].unique())
label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}

print(f"Loaded Train Sequences: {train_df['sequence_id'].nunique()}")
print(f"Loaded Test Sequences: {test_df['sequence_id'].nunique()}")
print(f"Total Classes: {len(unique_labels)}")

Loaded Train Sequences: 6515
Loaded Test Sequences: 1629
Total Classes: 18


## 3. Engineering for FFT-MLP Architecture

The first model described in the paper is a Multilayer Perceptron (MLP) that takes frequency-domain features as input.

### 3.1 FFT Transformation Logic
We define a helper function `compute_fft_features` that:
1. Groups the data by sequence.
2. Applies the Fast Fourier Transform (FFT) to the temporal dimension.
3. Computes the **magnitude** of the FFT components.
4. Resamples or pads the result to ensure fixed input size for the MLP.

In [13]:
def compute_fft_features(df, feature_cols, target_length=64):
    """
    Transforms time-domain sequences into fixed-length frequency-domain features.
    Returns:
        X: Array of shape (n_sequences, n_features * target_length)
        y: Array of labels (n_sequences,)
    """
    X_list = []
    y_list = []
    
    # Group by sequence
    grouped = df.groupby("sequence_id")
    
    for seq_id, group in grouped:
        # 1. Get raw time-series
        signals = group[feature_cols].values
        
        # 2. Apply 1D FFT along the time axis
        fft_complex = np.fft.rfft(signals, axis=0)
        
        # 3. Compute Magnitude
        fft_mag = np.abs(fft_complex)
        
        # 4. Pad/Truncate
        current_len = fft_mag.shape[0]
        if current_len < target_length:
            pad_width = ((0, target_length - current_len), (0, 0))
            fft_mag = np.pad(fft_mag, pad_width, mode='constant')
        else:
            fft_mag = fft_mag[:target_length, :]
            
        # 5. Flatten
        flat_features = fft_mag.flatten()
        
        X_list.append(flat_features)
        y_list.append(label_to_idx[group[LABEL_COL].iloc[0]])
        
    return np.array(X_list), np.array(y_list)

print("FFT Feature extraction function defined.")

FFT Feature extraction function defined.


### 3.2 Dataset Preparation
We apply the FFT transformation to both train and test sets using **IMU and Thermopile** sensors. We then apply Standardization (Zero Mean, Unit Variance) as required by the paper.

In [14]:
# Select features for the MLP (IMU + Thermopile)
mlp_features = IMU_COLS + THM_COLS

print(f"Extracting FFT features for {len(mlp_features)} sensors...")

# Compute features
X_train_fft, y_train_fft = compute_fft_features(train_df, mlp_features)
X_test_fft, y_test_fft = compute_fft_features(test_df, mlp_features)

# Apply Standardization
scaler = StandardScaler()
X_train_fft = scaler.fit_transform(X_train_fft)
X_test_fft = scaler.transform(X_test_fft)

print(f"FFT Training Data Shape: {X_train_fft.shape}")

Extracting FFT features for 12 sensors...
FFT Training Data Shape: (6515, 768)


### 3.3 PyTorch Dataset Class
We create a custom Dataset class and wrap our numpy arrays into DataLoaders with a batch size of 256.

In [15]:
class BFRB_Dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 256

train_dataset_mlp = BFRB_Dataset(X_train_fft, y_train_fft)
test_dataset_mlp = BFRB_Dataset(X_test_fft, y_test_fft)

train_loader_mlp = DataLoader(train_dataset_mlp, batch_size=BATCH_SIZE, shuffle=True)
test_loader_mlp = DataLoader(test_dataset_mlp, batch_size=BATCH_SIZE, shuffle=False)

print("MLP DataLoaders ready.")

MLP DataLoaders ready.


## 4. MLP Model Architecture
We define the Multilayer Perceptron described in the paper with 3 hidden layers (128, 64, 64 neurons).

In [17]:
class BFRB_MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(BFRB_MLP, self).__init__()
        
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes) 
        )
        
    def forward(self, x):
        return self.layers(x)

## 5. Training Loop
We define a reusable function `train_model` that handles the training process, loss calculation, and validation accuracy reporting.

In [18]:
def train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=20):
    model = model.to(DEVICE)
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {running_loss/len(train_loader):.4f} | Val Acc: {100*val_correct/val_total:.2f}%")
              
    return model

In [19]:
# Initialize Model
input_dim_mlp = X_train_fft.shape[1] 
num_classes = len(unique_labels)
mlp_model = BFRB_MLP(input_dim_mlp, num_classes)

# Define Optimization (Adam, lr=0.001 per paper)
criterion = nn.CrossEntropyLoss()
optimizer_mlp = optim.Adam(mlp_model.parameters(), lr=0.001)

# Train
print("Starting MLP Training...")
mlp_model = train_model(mlp_model, train_loader_mlp, test_loader_mlp, criterion, optimizer_mlp, num_epochs=20)
print("MLP Training Complete.")

Starting MLP Training...
Epoch [5/20] Loss: nan | Val Acc: 7.98%
Epoch [10/20] Loss: nan | Val Acc: 7.98%
Epoch [15/20] Loss: nan | Val Acc: 7.98%
Epoch [20/20] Loss: nan | Val Acc: 7.98%
MLP Training Complete.


## 6. Engineering for CNN-BiLSTM (TOF Data)
The second model uses Time-of-Flight (TOF) data. We must reshape the 320 flattened columns into $8 \times 8$ matrices with 5 channels (one for each sensor).

In [20]:
def prepare_tof_data(df, tof_cols, label_map, target_length=64):
    """
    Reshapes TOF data for CNN input.
    Output Shape: (N_Sequences, Time, Channels=5, Height=8, Width=8)
    """
    X_list = []
    y_list = []
    
    grouped = df.groupby("sequence_id")
    
    for seq_id, group in grouped:
        raw_tof = group[tof_cols].values
        
        # Pad/Truncate time
        current_len = raw_tof.shape[0]
        if current_len < target_length:
            pad_width = ((0, target_length - current_len), (0, 0))
            raw_tof = np.pad(raw_tof, pad_width, mode='constant', constant_values=-1)
        else:
            raw_tof = raw_tof[:target_length, :]
            
        # Reshape to 5 sensors x 8x8 grid
        reshaped_tof = raw_tof.reshape(target_length, 5, 8, 8)
        reshaped_tof[reshaped_tof == -1] = 0 # Handle missing values
        
        X_list.append(reshaped_tof)
        y_list.append(label_map[group[LABEL_COL].iloc[0]])
        
    return np.array(X_list), np.array(y_list)

print("Processing TOF Data...")
X_train_tof, y_train_tof = prepare_tof_data(train_df, TOF_COLS, label_to_idx)
X_test_tof, y_test_tof = prepare_tof_data(test_df, TOF_COLS, label_to_idx)

# Create Loaders
train_loader_tof = DataLoader(BFRB_Dataset(X_train_tof, y_train_tof), batch_size=256, shuffle=True)
test_loader_tof = DataLoader(BFRB_Dataset(X_test_tof, y_test_tof), batch_size=256, shuffle=False)
print("TOF Loaders ready.")

Processing TOF Data...
TOF Loaders ready.


## 7. CNN-BiLSTM Architecture
This model uses a 2D CNN to extract spatial features from the $8 \times 8$ TOF grids, followed by a Bidirectional LSTM for temporal learning.

In [21]:
class BFRB_CNN_BiLSTM(nn.Module):
    def __init__(self, num_classes):
        super(BFRB_CNN_BiLSTM, self).__init__()
        
        # CNN for Spatial Features
        self.cnn = nn.Sequential(
            nn.Conv2d(5, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
        
        self.projection = nn.Linear(64 * 8 * 8, 128)
        
        # BiLSTM for Temporal Features
        self.lstm = nn.LSTM(input_size=128, hidden_size=128, 
                            num_layers=1, batch_first=True, bidirectional=True)
        
        self.classifier = nn.Linear(256, num_classes)
        
    def forward(self, x):
        batch_size, time_steps, C, H, W = x.size()
        
        # Fold time into batch for CNN
        c_in = x.view(batch_size * time_steps, C, H, W)
        c_out = self.cnn(c_in)
        c_out = self.projection(c_out)
        
        # Unfold time for LSTM
        r_in = c_out.view(batch_size, time_steps, -1)
        lstm_out, _ = self.lstm(r_in)
        
        # Take last timestep
        final_feature = lstm_out[:, -1, :] 
        return self.classifier(final_feature)

In [22]:
# Initialize
cnn_lstm_model = BFRB_CNN_BiLSTM(num_classes)

# Optimizer: AdamW (lr=0.001, weight_decay=0.0001 per paper)
optimizer_cnn = optim.AdamW(cnn_lstm_model.parameters(), lr=0.001, weight_decay=0.0001)

# Train
print("Starting CNN-BiLSTM Training...")
cnn_lstm_model = train_model(cnn_lstm_model, train_loader_tof, test_loader_tof, criterion, optimizer_cnn, num_epochs=20)

# Save Models
torch.save(mlp_model.state_dict(), DATA_PROCESSED / "model_mlp_paper.pth")
torch.save(cnn_lstm_model.state_dict(), DATA_PROCESSED / "model_cnn_lstm_paper.pth")

# Save Label Map for Evaluation
with open(DATA_PROCESSED / "label_map.pkl", "wb") as f:
    pickle.dump(label_to_idx, f)

print("All models saved successfully.")

Starting CNN-BiLSTM Training...
Epoch [5/20] Loss: nan | Val Acc: 7.98%
Epoch [10/20] Loss: nan | Val Acc: 7.98%
Epoch [15/20] Loss: nan | Val Acc: 7.98%
Epoch [20/20] Loss: nan | Val Acc: 7.98%
All models saved successfully.
